# Inflation Forecasting with LSTM (RNN)

Forecasts **1-month-ahead PCE inflation (PCEPI)** using an LSTM neural network trained on FRED-MD data.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from fred_md_utils import (
    download_latest_vintage,
    get_latest_vintage,
    build_dataset_from_csv,
    make_sequences,
)

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# ── Parameters ──────────────────────────────────────────────────────────────
SEQ_LEN       = 24    # months of history fed to the LSTM at each step
LSTM_UNITS    = 64    # hidden units in the LSTM layer
DROPOUT       = 0.2   # recurrent dropout rate
EPOCHS        = 200   # max training epochs (EarlyStopping will cut this short)
BATCH_SIZE    = 16
PATIENCE      = 20    # EarlyStopping patience
LEARNING_RATE = 1e-3

TEST_START = '2025-06-01'
VAL_START  = '2023-01-01'

# Paths (works from notebooks/ or repo root)
VINTAGE_DIR = '../data' if os.path.exists('../data') else 'data'

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {bool(tf.config.list_physical_devices('GPU'))}")
print(f"Vintage dir        : {os.path.abspath(VINTAGE_DIR)}")


## Load & Split Data

In [ ]:
# Download (or locate) the most recent FRED-MD vintage
vintage_file = download_latest_vintage(VINTAGE_DIR)

# Build 2D train/val/test splits — no lag columns (n_lags=0)
# The LSTM captures temporal dependencies through sequences instead.
X_train, y_train, X_val, y_val, X_test, y_test, feature_names = build_dataset_from_csv(
    filepath=vintage_file,
    horizon=1,
    n_lags=0,           # no manual lag columns — LSTM uses sequences
    test_start=TEST_START,
    val_start=VAL_START,
)

print(f"\nBase features : {len(feature_names)}")
print(f"Sequence length: {SEQ_LEN} months")
print(f"Input shape will be: (batch, {SEQ_LEN}, {len(feature_names)})")


## Scale Features

In [ ]:
# Fit StandardScaler on training features only; transform val and test.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.values)
X_val_scaled   = scaler.transform(X_val.values)
X_test_scaled  = scaler.transform(X_test.values)

y_scaler = StandardScaler()
y_train_arr = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_val_arr   = y_scaler.transform(y_val.values.reshape(-1, 1)).ravel()
y_test_arr  = y_test.values   # raw — predictions inverse-transformed before eval


## Build LSTM Sequences

In [ ]:
# Training sequences: straightforward — no cross-split context needed.
X_train_seq, y_train_seq = make_sequences(X_train_scaled, y_train_arr, SEQ_LEN)

# Val sequences: prepend the last (SEQ_LEN-1) training rows so the first
# val sample has a full context window.
# After prepending (SEQ_LEN-1) rows, make_sequences produces exactly len(X_val)
# sequences — one per val observation — so no further slicing is needed.
X_val_seq, y_val_seq = make_sequences(
    np.vstack([X_train_scaled[-(SEQ_LEN - 1):], X_val_scaled]),
    np.concatenate([y_train_arr[-(SEQ_LEN - 1):], y_val_arr]),
    SEQ_LEN,
)

# Test sequences: same logic, prepend last (SEQ_LEN-1) val rows.
X_test_seq, y_test_seq = make_sequences(
    np.vstack([X_val_scaled[-(SEQ_LEN - 1):], X_test_scaled]),
    np.concatenate([y_val_arr[-(SEQ_LEN - 1):], y_test_arr]),
    SEQ_LEN,
)

print(f"Train sequences : {X_train_seq.shape}  targets: {y_train_seq.shape}")
print(f"Val   sequences : {X_val_seq.shape}  targets: {y_val_seq.shape}")
print(f"Test  sequences : {X_test_seq.shape}  targets: {y_test_seq.shape}")


## Define RNNForecaster

In [ ]:
class RNNForecaster:
    """
    LSTM-based inflation forecaster.

    Accepts 3-D input (batch, seq_len, n_features) produced by make_sequences().
    Exposes the same evaluate() interface as InflationForecaster so metrics are
    directly comparable across notebooks.
    """

    def __init__(self, seq_len, n_features,
                 units=LSTM_UNITS, dropout=DROPOUT, lr=LEARNING_RATE,
                 y_scaler=None):
        self.seq_len    = seq_len
        self.n_features = n_features
        self.units      = units
        self.dropout    = dropout
        self.lr         = lr
        self.y_scaler   = y_scaler
        self.model      = self._build_model()
        self.history    = None

    def _build_model(self):
        model = keras.Sequential([
            keras.layers.Input(shape=(self.seq_len, self.n_features)),
            keras.layers.LSTM(self.units, dropout=self.dropout,
                              recurrent_dropout=0.0),
            keras.layers.Dense(32, activation='relu'),
            keras.layers.Dense(1),
        ])
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=self.lr),
            loss='mse',
        )
        return model

    def train(self, X_train_seq, y_train_seq,
              X_val_seq, y_val_seq,
              epochs=EPOCHS, batch_size=BATCH_SIZE, patience=PATIENCE):
        early_stop = keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=patience,
            restore_best_weights=True,
            verbose=1,
        )
        self.history = self.model.fit(
            X_train_seq, y_train_seq,
            validation_data=(X_val_seq, y_val_seq),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=[early_stop],
            verbose=1,
        )
        return self

    def predict(self, X_seq):
        """Accept 3-D input (n_samples, seq_len, n_features), return 1-D predictions
        in original (unscaled) target units if y_scaler was provided."""
        raw = self.model.predict(X_seq, verbose=0).flatten()
        if self.y_scaler is not None:
            return self.y_scaler.inverse_transform(raw.reshape(-1, 1)).ravel()
        return raw

    def evaluate(self, X_seq, y_true):
        """
        Returns (metrics_dict, y_pred).
        Metric names and scale are identical to InflationForecaster.evaluate().
        """
        y_pred = self.predict(X_seq)
        y_true = np.asarray(y_true)
        metrics = {
            'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
            'mae':  float(mean_absolute_error(y_true, y_pred)),
            'r2':   float(r2_score(y_true, y_pred)),
            'mape': float(np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100),
        }
        return metrics, y_pred

forecaster = RNNForecaster(
    seq_len=SEQ_LEN,
    n_features=len(feature_names),
    units=LSTM_UNITS,
    dropout=DROPOUT,
    lr=LEARNING_RATE,
    y_scaler=y_scaler,
)
forecaster.model.summary()


## Train

In [ ]:
forecaster.train(
    X_train_seq, y_train_seq,
    X_val_seq,   y_val_seq,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    patience=PATIENCE,
)


## Training & Validation Loss

In [ ]:
hist = forecaster.history.history
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hist['loss'],     label='Train MSE', linewidth=1.5)
ax.plot(hist['val_loss'], label='Val MSE',   linewidth=1.5, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.set_title('LSTM Training History')
ax.legend()
best_epoch = int(np.argmin(hist['val_loss'])) + 1
ax.axvline(best_epoch, color='crimson', linestyle=':', linewidth=1.2,
           label=f'Best epoch ({best_epoch})')
ax.legend()
fig.tight_layout()
plt.show()
print(f"Best epoch: {best_epoch}  |  Best val MSE: {min(hist['val_loss']):.6f}")


## Evaluate on Test Set

In [ ]:
test_metrics, y_test_pred = forecaster.evaluate(X_test_seq, y_test_seq)

print("Out-of-Sample Test Performance (LSTM):")
print(f"  Test date range : {y_test.index.min().date()} to {y_test.index.max().date()}")
print(f"  n observations  : {len(y_test_seq)}")
print(f"  RMSE : {test_metrics['rmse']:.4f}")
print(f"  MAE  : {test_metrics['mae']:.4f}")
print(f"  R\u00b2   : {test_metrics['r2']:.4f}")
print(f"  MAPE : {test_metrics['mape']:.2f}%")
print()
print("Metric scale: monthly log-diff \u00d7 100 (percentage points of monthly PCE inflation)")


## Predictions vs Actuals

In [ ]:
# y_test_seq is aligned to y_test.index (all test observations)
test_dates = y_test.index

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(test_dates, y_test_seq,  'o-', label='Actual',          linewidth=1.8, markersize=5, color='steelblue')
ax.plot(test_dates, y_test_pred, 's--', label='LSTM Forecast',  linewidth=1.8, markersize=5, color='darkorange', alpha=0.85)
ax.axhline(0, color='grey', linewidth=0.5)
ax.set_title(f'LSTM: Out-of-Sample Test Predictions (seq_len={SEQ_LEN})')
ax.set_ylabel('PCEPI Monthly Growth (log diff \u00d7 100)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.tight_layout()
plt.show()


## Data-Anchored & Chained Forecasts

In [ ]:
import datetime
from dateutil.relativedelta import relativedelta
from fred_md_utils import load_fred_md_file, load_fred_md_raw

# --- Load full transformed vintage for anchor lookups ---
df_feat, _ = load_fred_md_file(vintage_file)
df_feat = df_feat.ffill().bfill()

TARGET_FEATURE   = 'PCEPI'
pcepi_idx        = feature_names.index(TARGET_FEATURE)
anchor_date      = df_feat.index.max()
current_month    = pd.Timestamp(datetime.date.today().replace(day=1))
chain_start_date = y_test.index.max()   # last test observation

total_forecast_steps = (
    (current_month.year  - chain_start_date.year)  * 12 +
    (current_month.month - chain_start_date.month)
)

print(f"Vintage anchor      : {anchor_date.date()}")
print(f"Chain start         : {chain_start_date.date()}")
print(f"Current month       : {current_month.date()}")
print(f"Total forecast steps: {total_forecast_steps}")

# --- Train medians as fallback for NaN-filled vintage rows ---
train_medians = X_train.median()

# --- Initialize buffer: last SEQ_LEN unscaled rows up to chain_start_date ---
# Combine val + test unscaled features; take last SEQ_LEN rows (ending at chain_start_date).
X_hist_unscaled = np.vstack([X_val.values, X_test.values])
buffer_unscaled  = X_hist_unscaled[-SEQ_LEN:].copy()   # (SEQ_LEN, n_features)

# PCEPI monthly growth from vintage (for YoY calculation and anchored updates)
# Load raw levels to compute PCEPI growth as first log-diff × 100 (tcode 5),
# matching the training target — FRED-MD native tcode for PCEPI is 6 (second diff).
df_raw_vintage, _ = load_fred_md_raw(vintage_file)
pcepi_growth = (np.log(df_raw_vintage['PCEPI']).diff() * 100).ffill().bfill()

# --- Forecast loop ---
forecast_rows = []

for step in range(1, total_forecast_steps + 1):
    # Scale buffer at prediction time, reshape, and predict
    buf_scaled = scaler.transform(buffer_unscaled)                 # (SEQ_LEN, n_features)
    X_seq      = buf_scaled.reshape(1, SEQ_LEN, len(feature_names))
    pred       = float(forecaster.predict(X_seq)[0])

    forecast_date = chain_start_date + pd.DateOffset(months=step)
    is_anchored   = forecast_date in df_feat.index

    # YoY: rolling 12-month sum of monthly log-diffs
    prior_preds     = {r['date']: r['forecast'] for r in forecast_rows}
    monthly_growths = []
    for i in range(12):
        month = forecast_date - pd.DateOffset(months=11 - i)
        if month == forecast_date:
            monthly_growths.append(pred)
        elif month in pcepi_growth.index and pd.notna(pcepi_growth.loc[month]):
            monthly_growths.append(float(pcepi_growth.loc[month]))
        elif month in prior_preds:
            monthly_growths.append(prior_preds[month])
        else:
            monthly_growths = None
            break
    yoy = sum(monthly_growths) if monthly_growths is not None else float('nan')

    forecast_rows.append({
        'step':             step,
        'date':             forecast_date,
        'forecast':         pred,
        'annualized_naive': pred * 12,
        'yoy_equivalent':   yoy,
        'anchored':         is_anchored,
    })

    # Update buffer: use actual vintage row if available, else chain PCEPI from pred
    if is_anchored:
        new_row = (df_feat.reindex(columns=feature_names)
                          .loc[forecast_date]
                          .fillna(train_medians)
                          .values.copy())
    else:
        new_row             = buffer_unscaled[-1].copy()
        new_row[pcepi_idx]  = pred   # advance PCEPI to predicted monthly growth

    buffer_unscaled = np.vstack([buffer_unscaled[1:], new_row.reshape(1, -1)])

forecast_df = pd.DataFrame(forecast_rows)

# Print results table
print(f"\n{'='*70}")
print('  LATEST FORECASTS (LSTM)')
print(f"  Vintage anchor: {anchor_date.date()}  |  Chain start: {chain_start_date.date()}")
for row in forecast_rows:
    month_label = row['date'].strftime('%b %Y')
    tag     = '(data-anchored)' if row['anchored'] else '(chained)     '
    yoy_str = (f"{row['yoy_equivalent']:+.2f}% YoY"
               if not np.isnan(row['yoy_equivalent']) else 'YoY: N/A')
    print(f"  {month_label} (step {row['step']:2d}) {tag}: "
          f"{row['forecast']:+.4f}% monthly  ({yoy_str})")
print(f"{'='*70}")


## Forecast Plot

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

# Test actuals + LSTM test predictions
ax.plot(y_test.index, y_test_seq,  'o-', label='Actual (test)',
        linewidth=1.8, markersize=5, color='steelblue')
ax.plot(y_test.index, y_test_pred, 's--', label='LSTM Forecast (test)',
        linewidth=1.8, alpha=0.8, markersize=5, color='darkorange')

# Connector from last test obs to first forecast step
ax.plot([y_test.index[-1], forecast_df.iloc[0]['date']],
        [y_test_seq[-1],   forecast_df.iloc[0]['forecast']],
        '--', color='crimson', linewidth=1.2, alpha=0.6)

# Data-anchored vs chained forecast segments
anchored_df = forecast_df[forecast_df['anchored']]
chained_df  = forecast_df[~forecast_df['anchored']]

if not anchored_df.empty:
    ax.plot(anchored_df['date'], anchored_df['forecast'],
            'o-', color='crimson', linewidth=1.8, alpha=0.9, markersize=6,
            label='Forecast (data-anchored)')
if not chained_df.empty:
    ax.plot(chained_df['date'], chained_df['forecast'],
            's--', color='crimson', linewidth=1.8, alpha=0.8, markersize=6,
            label='Forecast (chained)')
if not anchored_df.empty and not chained_df.empty:
    ax.plot([anchored_df.iloc[-1]['date'], chained_df.iloc[0]['date']],
            [anchored_df.iloc[-1]['forecast'], chained_df.iloc[0]['forecast']],
            '-', color='crimson', linewidth=1.8, alpha=0.8)

# Star on current-month forecast
last_row  = forecast_df.iloc[-1]
yoy_label = (f"\u2248 {last_row['yoy_equivalent']:+.1f}% YoY"
             if not np.isnan(last_row['yoy_equivalent']) else '')
ax.plot(last_row['date'], last_row['forecast'],
        '*', color='crimson', markersize=18, zorder=5,
        label=f"Current Month Forecast ({last_row['date'].strftime('%b %Y')})")
ax.annotate(
    f"{last_row['date'].strftime('%b %Y')}\n{last_row['forecast']:+.3f}% monthly\n{yoy_label}",
    xy=(last_row['date'], last_row['forecast']),
    xytext=(24, 35), textcoords='offset points', fontsize=9, color='crimson',
    arrowprops=dict(arrowstyle='->', color='crimson', lw=1.4),
    bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='crimson', lw=1.1),
)

ax.axhline(0, color='grey', linewidth=0.5)
ax.set_title(f'LSTM: Out-of-Sample Test + Forecasts '
             f'(vintage thru {anchor_date.strftime("%b %Y")}, seq_len={SEQ_LEN})')
ax.set_ylabel('PCEPI Monthly Growth (log diff \u00d7 100)')
ax.legend(loc='upper left', fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.set_xlim(left=y_test.index.min() - pd.DateOffset(months=1),
            right=forecast_df['date'].max() + pd.DateOffset(months=1))
fig.tight_layout()
plt.show()


## Save Predictions for Model Comparison

In [ ]:
# Save test predictions and aligned dates for model_comparison.ipynb.
os.makedirs('results', exist_ok=True)

test_dates_seq = y_test.index   # fully aligned — one prediction per test obs

np.save('results/rnn_preds.npy',       y_test_pred)
np.save('results/rnn_dates.npy',       np.array(test_dates_seq, dtype='datetime64[ns]'))
np.save('results/rnn_actuals.npy',     y_test_seq)

print(f"Saved {len(y_test_pred)} test predictions to results/")
print(f"  Date range: {test_dates_seq.min().date()} to {test_dates_seq.max().date()}")
